In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model, lr_init=0.001, line_search_method="const", line_search_cond="armijo")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.5834920406341553
epoch:  1, loss: 0.5802018046379089
epoch:  2, loss: 0.5769302248954773
epoch:  3, loss: 0.5736771821975708
epoch:  4, loss: 0.5704424381256104
epoch:  5, loss: 0.5672258138656616
epoch:  6, loss: 0.5640273690223694
epoch:  7, loss: 0.5608469843864441
epoch:  8, loss: 0.5576845407485962
epoch:  9, loss: 0.5545400977134705
epoch:  10, loss: 0.5514131784439087
epoch:  11, loss: 0.54830402135849
epoch:  12, loss: 0.5452125668525696
epoch:  13, loss: 0.5421382784843445
epoch:  14, loss: 0.5390810966491699
epoch:  15, loss: 0.5360410213470459
epoch:  16, loss: 0.5330178737640381
epoch:  17, loss: 0.5300114750862122
epoch:  18, loss: 0.5270219445228577
epoch:  19, loss: 0.5240491628646851
epoch:  20, loss: 0.5210928916931152
epoch:  21, loss: 0.5181534290313721
epoch:  22, loss: 0.5152302980422974
epoch:  23, loss: 0.5123234987258911
epoch:  24, loss: 0.5094330310821533
epoch:  25, loss: 0.5065585970878601
epoch:  26, loss: 0.5037001967430115
epoch:  27, l

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -387.6046142578125
Test metrics:  R2 = -456.5317077636719


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model, lr=1, line_search_method="backtrack", line_search_cond="armijo")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.6700921654701233
epoch:  1, loss: 0.38180243968963623
epoch:  2, loss: 0.2267121821641922
epoch:  3, loss: 0.14017057418823242
epoch:  4, loss: 0.09181701391935349
epoch:  5, loss: 0.06497926265001297
epoch:  6, loss: 0.05022768676280975
epoch:  7, loss: 0.04221273213624954
epoch:  8, loss: 0.037904806435108185
epoch:  9, loss: 0.03561016544699669
epoch:  10, loss: 0.03439651057124138
epoch:  11, loss: 0.03375780209898949
epoch:  12, loss: 0.03342262655496597
epoch:  13, loss: 0.03324681147933006
epoch:  14, loss: 0.03315430134534836
epoch:  15, loss: 0.033105261623859406
epoch:  16, loss: 0.03307879716157913
epoch:  17, loss: 0.03306405618786812
epoch:  18, loss: 0.0330553874373436
epoch:  19, loss: 0.033049825578927994
epoch:  20, loss: 0.03303675726056099
epoch:  21, loss: 0.03302907943725586
epoch:  22, loss: 0.033024862408638
epoch:  23, loss: 0.03301355615258217
epoch:  24, loss: 0.0330069400370121
epoch:  25, loss: 0.03300373628735542
epoch:  26, loss: 0.03299

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7346940040588379
Test metrics:  R2 = 0.7563313841819763


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model, lr_init=1, line_search_method="interpolate", line_search_cond="armijo")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.5306410193443298
epoch:  1, loss: 0.03386431559920311
epoch:  2, loss: 0.033309414982795715
epoch:  3, loss: 0.03323952481150627
epoch:  4, loss: 0.03318201005458832
epoch:  5, loss: 0.033129021525382996
epoch:  6, loss: 0.033077891916036606
epoch:  7, loss: 0.033027585595846176
epoch:  8, loss: 0.032982904464006424
epoch:  9, loss: 0.032948996871709824
epoch:  10, loss: 0.03293236717581749
epoch:  11, loss: 0.03282274678349495
epoch:  12, loss: 0.032767150551080704
epoch:  13, loss: 0.03271268680691719
epoch:  14, loss: 0.032660115510225296
epoch:  15, loss: 0.032614465802907944
epoch:  16, loss: 0.032579001039266586
epoch:  17, loss: 0.032560210675001144
epoch:  18, loss: 0.03247442469000816
epoch:  19, loss: 0.03242863714694977
epoch:  20, loss: 0.03238461911678314
epoch:  21, loss: 0.032344017177820206
epoch:  22, loss: 0.03230871632695198
epoch:  23, loss: 0.03228271007537842
epoch:  24, loss: 0.03227490186691284
epoch:  25, loss: 0.0321742445230484
epoch:  26, 

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7325795292854309
Test metrics:  R2 = 0.7723909616470337


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model, lr_init=1, line_search_method="bisect", line_search_cond="armijo")

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.07874749600887299
epoch:  1, loss: 0.041627731174230576
epoch:  2, loss: 0.03417794778943062
epoch:  3, loss: 0.03309084475040436
epoch:  4, loss: 0.0329345166683197
epoch:  5, loss: 0.03291098400950432
epoch:  6, loss: 0.03290415182709694
epoch:  7, loss: 0.03289731964468956
epoch:  8, loss: 0.03289082646369934
epoch:  9, loss: 0.03288505598902702
epoch:  10, loss: 0.032878875732421875
epoch:  11, loss: 0.03287165239453316
epoch:  12, loss: 0.03286471590399742
epoch:  13, loss: 0.032858386635780334
epoch:  14, loss: 0.03285326436161995
epoch:  15, loss: 0.03284492716193199
epoch:  16, loss: 0.03283745422959328
epoch:  17, loss: 0.03283054381608963
epoch:  18, loss: 0.03282487019896507
epoch:  19, loss: 0.032817382365465164
epoch:  20, loss: 0.032809849828481674
epoch:  21, loss: 0.03280266374349594
epoch:  22, loss: 0.032796427607536316
epoch:  23, loss: 0.03278922662138939
epoch:  24, loss: 0.03278123587369919
epoch:  25, loss: 0.032773569226264954
epoch:  26, loss

In [12]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7938903570175171
Test metrics:  R2 = 0.7873958349227905
